# 01 — Data Cleaning
**Pi Systems Private Limited**  
Fictional company and synthetic dataset created for portfolio demonstration.

This notebook demonstrates inspection, missing values, duplicates, date normalization, master-data reconciliation, financial recalculation, and validation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
BASE = Path("..")
raw = pd.read_csv(BASE / "data/raw/01_raw_data.csv")
raw.head()


In [ ]:
raw.tail(); raw.shape; raw.columns.tolist()


In [ ]:
raw.info(); raw.describe(include="all")


In [ ]:
raw.isnull().sum().sort_values(ascending=False).head(15)


In [ ]:
raw.duplicated().sum(); raw.nunique().sort_values().head(15)


## Why these checks?
`head()`/`tail()` verify structure, `info()` checks types and nullability, `describe()` reveals range/outliers, `isnull()` quantifies missingness, `duplicated()` finds repeated rows, and `nunique()`/`value_counts()` expose inconsistent categories.


In [ ]:
raw["Category"].value_counts(dropna=False).head(20)


In [ ]:
# Use the production cleaning script so notebook and pipeline remain consistent
%run ../python/clean_data.py


In [ ]:
clean = pd.read_csv(BASE / "data/processed/02_cleaned_sales_data.csv", parse_dates=["Order_Date"])
clean.info(); clean.isnull().sum().sum(); clean.duplicated().sum()


In [ ]:
# Outlier investigation: retain valid high-value enterprise transactions
q1, q3 = clean["Sales_Amount"].quantile([.25,.75]); iqr=q3-q1
outliers = clean.loc[clean.Sales_Amount > q3 + 1.5*iqr]
outliers[["Customer_Segment","Quantity","Sales_Amount"]].describe(include="all")


In [ ]:
# Validation
assert (clean.Quantity >= 0).all()
assert (clean.Unit_Price >= 0).all()
assert np.allclose(clean.Sales_Amount, clean.Gross_Sales-clean.Discount_Amount, atol=.02)
assert np.allclose(clean.Profit, clean.Sales_Amount-clean.Cost_Amount, atol=.02)
print("Validation passed")
